In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("tabla_source", "customers_silver")
dbutils.widgets.text("esquema_sink", "gold")

dbutils.widgets.text("fact_table", "dim_customers")
dbutils.widgets.text("agg_gender_table", "agg_customers_by_gender")
dbutils.widgets.text("agg_segment_table", "agg_customers_by_segmentation")
dbutils.widgets.text("agg_age_table", "agg_customers_by_age_group")
dbutils.widgets.text("kpi_table", "kpi_customers")

catalogo = dbutils.widgets.get("catalogo")
esq_src = dbutils.widgets.get("esquema_source")
tab_src = dbutils.widgets.get("tabla_source")
esq_sink = dbutils.widgets.get("esquema_sink")

fact_table = dbutils.widgets.get("fact_table")
agg_gender_table = dbutils.widgets.get("agg_gender_table")
agg_segment_table = dbutils.widgets.get("agg_segment_table")
agg_age_table = dbutils.widgets.get("agg_age_table")
kpi_table = dbutils.widgets.get("kpi_table")

In [0]:
df_silver = spark.table(f"{catalogo}.{esq_src}.{tab_src}")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

fact_df = df_silver.select(
    "ID",
    "Gender",
    "Age",
    "Age_Group",
    "Profession",
    "Work_Experience",
    "Family_Size",
    "Spending_Score",
    "Spending_Level_Score",
    "Segmentation",
    "Ever_Married_Flag",
    "Graduated_Flag"
)

fact_df.write.mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esq_sink}.{fact_table}")

In [0]:
df_silver.groupBy("Gender").agg(
    count("*").alias("Total_Customers"),
    avg("Age").alias("Avg_Age"),
    avg("Work_Experience").alias("Avg_Work_Experience"),
    avg("Spending_Level_Score").alias("Avg_Spending_Score")
).write.mode("overwrite") \
 .saveAsTable(f"{catalogo}.{esq_sink}.{agg_gender_table}")

In [0]:
df_silver.groupBy("Segmentation").agg(
    count("*").alias("Total_Customers"),
    avg("Age").alias("Avg_Age"),
    avg("Spending_Level_Score").alias("Avg_Spending_Score")
).write.mode("overwrite") \
 .saveAsTable(f"{catalogo}.{esq_sink}.{agg_segment_table}")

In [0]:
df_silver.groupBy("Age_Group").agg(
    count("*").alias("Total_Customers"),
    avg("Work_Experience").alias("Avg_Work_Experience"),
    avg("Spending_Level_Score").alias("Avg_Spending_Score")
).write.mode("overwrite") \
 .saveAsTable(f"{catalogo}.{esq_sink}.{agg_age_table}")

In [0]:
df_silver.agg(
    count("*").alias("Total_Customers"),
    avg("Age").alias("Avg_Age"),
    avg("Work_Experience").alias("Avg_Work_Experience"),
    avg("Spending_Level_Score").alias("Avg_Spending_Level")
).write.mode("overwrite") \
 .saveAsTable(f"{catalogo}.{esq_sink}.{kpi_table}")